In [6]:
import cv2
from pathlib import Path

# project root = parent of notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

video = ROOT / "evaluation_videos/cctv/Camera1.mp4"
print("Looking for:", video, "| exists:", video.exists())

cap = cv2.VideoCapture(str(video))
cap.set(cv2.CAP_PROP_POS_FRAMES, 30)
ret, frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("Could not read frame — check the path above")

out = ROOT / "outputs/calib_frame.jpg"
out.parent.mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(out), frame)
print("Saved:", out, "| Shape:", frame.shape)

Looking for: /Users/aayushaswal/aisleguard/evaluation_videos/cctv/Camera1.mp4 | exists: True
Saved: /Users/aayushaswal/aisleguard/outputs/calib_frame.jpg | Shape: (1080, 1920, 3)


In [ ]:
import cv2

img = cv2.imread(str(out))   # 'out' from the previous cell = calib_frame.jpg
clicks = []

def on_click(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        clicks.append((x, y))
        print(f"Point {len(clicks)}: ({x}, {y})")
        cv2.circle(img, (x, y), 8, (0, 0, 255), -1)
        cv2.putText(img, str(len(clicks)), (x+10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        cv2.imshow("Click 4 points: P1 near-left, P2 near-right, P3 far-left, P4 far-right", img)

cv2.imshow("Click 4 points: P1 near-left, P2 near-right, P3 far-left, P4 far-right", img)
cv2.setMouseCallback("Click 4 points: P1 near-left, P2 near-right, P3 far-left, P4 far-right", on_click)
cv2.waitKey(0)
cv2.destroyAllWindows()

print("\nPixel points:", clicks)

In [ ]:
import numpy as np
import cv2

# Pixel points, reordered to: near-left, near-right, far-left, far-right
pixel_pts = np.array([
    [734, 1049],   # near-left
    [1278, 1039],  # near-right
    [913, 66],     # far-left
    [1123, 60],    # far-right
], dtype=np.float32)

# Real-world metric rectangle (metres): X = across aisle, Y = down aisle
# aisle width 3.5 m, aisle length 11 m (estimated, synthetic video)
AISLE_W = 3.5
AISLE_L = 11.0
world_pts = np.array([
    [0,       0],        # near-left
    [AISLE_W, 0],        # near-right
    [0,       AISLE_L],  # far-left
    [AISLE_W, AISLE_L],  # far-right
], dtype=np.float32)

# Compute homography: pixel -> metres
H, status = cv2.findHomography(pixel_pts, world_pts)
print("Homography matrix H:\n", H)

# Reprojection check: push the pixel points through H, compare to true world coords
projected = cv2.perspectiveTransform(pixel_pts.reshape(-1, 1, 2), H).reshape(-1, 2)
print("\nReprojection check (metres):")
for i, (proj, true) in enumerate(zip(projected, world_pts)):
    err_cm = np.linalg.norm(proj - true) * 100
    print(f"  P{i+1}: projected=({proj[0]:.2f}, {proj[1]:.2f})  "
          f"true=({true[0]:.1f}, {true[1]:.1f})  error={err_cm:.1f} cm")

mean_err = np.mean([np.linalg.norm(p - t) for p, t in zip(projected, world_pts)]) * 100
print(f"\nMean reprojection error: {mean_err:.1f} cm")

Homography matrix H:
 [[ 1.99660441e-02  3.63572930e-03 -1.84689564e+01]
 [-2.45429537e-04 -1.33513668e-02  1.41857290e+01]
 [ 6.75460368e-05  1.93126283e-03  1.00000000e+00]]

Reprojection check (metres):
  P1: projected=(0.00, -0.00)  true=(0.0, 0.0)  error=0.0 cm
  P2: projected=(3.50, -0.00)  true=(3.5, 0.0)  error=0.0 cm
  P3: projected=(-0.00, 11.00)  true=(0.0, 11.0)  error=0.0 cm
  P4: projected=(3.50, 11.00)  true=(3.5, 11.0)  error=0.0 cm

Mean reprojection error: 0.0 cm


In [ ]:
import numpy as np

def pixel_to_metres(px, py, H):
    pt = np.array([[[px, py]]], dtype=np.float32)
    out = cv2.perspectiveTransform(pt, H)[0][0]
    return out[0], out[1]

# The small pallet box in the middle of the aisle.
# From your calib frame it's centered horizontally, upper-middle of the aisle.
# Estimate its bottom-center pixel — adjust if you can read it more precisely:
box_px, box_py = 1110, 360    # rough bottom of the little box

x_m, y_m = pixel_to_metres(box_px, box_py, H)
print(f"Pallet box -> X={x_m:.2f} m across aisle, Y={y_m:.2f} m down aisle")

# Sanity checks:
print(f"\nAcross-aisle position: {x_m:.2f} m (aisle is 3.5 m wide, so ~1.75 = centered)")
print(f"Down-aisle position:   {y_m:.2f} m (aisle is 11 m long)")

Pallet box -> X=2.83 m across aisle, Y=5.14 m down aisle

Across-aisle position: 2.83 m (aisle is 3.5 m wide, so ~1.75 = centered)
Down-aisle position:   5.14 m (aisle is 11 m long)


In [14]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "outputs/trajectories/camera1_tracks.csv")

forklift = df[df["class"] == "forklift"].iloc[100]
fx, fy = forklift["cx"], forklift["cy"]
x_m, y_m = pixel_to_metres(fx, fy, H)
print(f"Forklift at pixel ({fx:.0f},{fy:.0f}) -> ({x_m:.2f} m, {y_m:.2f} m)")
print(f"Inside aisle? X should be 0-3.5: {'YES' if 0 <= x_m <= 3.5 else 'check'}")

Forklift at pixel (973,111) -> (1.06 m, 9.74 m)
Inside aisle? X should be 0-3.5: YES


In [15]:
import numpy as np
import yaml
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

config = {
    "camera_id": "camera_01",
    "source": "evaluation_videos/cctv/Camera1.mp4",
    "resolution": [1920, 1080],
    "homography": H.tolist(),
    "aisle_width_m": 3.5,
    "aisle_length_m": 11.0,
    "calibration_pixel_points": [[734,1049],[1278,1039],[913,66],[1123,60]],
    "calibration_note": "Synthetic scene; aisle dimensions estimated. "
                        "Absolute scale approximate, relative distances consistent.",
}

out = ROOT / "configs/camera_01.yaml"
with open(out, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)
print("Saved homography config to", out)

Saved homography config to /Users/aayushaswal/aisleguard/configs/camera_01.yaml
